In [ ]:
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.transform import Affine
import numpy as np

def river_mask_hybrid(input_shp, dem_path, output_tif, coverage_threshold=0.3):
    """
    A hybrid rasterization method bridging all_touched=True and False 
    (Supersampling Area-Ratio Method).
    
    :param coverage_threshold: Area ratio threshold (between 0.0 and 1.0).
                               0.0001 is close to all_touched=True (any touch counts)
                               0.5 means vector must cover >50% of the pixel area
                               0.9 is close to all_touched=False
    """
    # 1. Read DEM specifications
    with rasterio.open(dem_path) as dem_src:
        dem_meta = dem_src.meta.copy()
        dem_transform = dem_src.transform
        dem_crs = dem_src.crs
        dem_width = dem_src.width
        dem_height = dem_src.height
        
    # 2. Read and reproject vector data
    gdf = gpd.read_file(input_shp)
    if gdf.crs != dem_crs:
        gdf = gdf.to_crs(dem_crs)

    # 3. Create a supersampled grid (scale=10 means 1 original pixel is split into 10x10=100 sub-pixels)
    scale = 10
    fine_width = dem_width * scale
    fine_height = dem_height * scale
    
    # Construct the high-resolution transform matrix
    fine_transform = Affine(
        dem_transform.a / scale, dem_transform.b, dem_transform.c,
        dem_transform.d, dem_transform.e / scale, dem_transform.f
    )
    
    # 4. Standard rasterization on the micro-grid (using False to capture edges with high precision)
    shapes = ((geom, 1) for geom in gdf.geometry)
    fine_mask = features.rasterize(
        shapes=shapes,
        out_shape=(fine_height, fine_width),
        transform=fine_transform,
        fill=0,
        all_touched=False, # Use False on the fine grid to approximate true geometric boundaries
        dtype='uint8'
    )
    
    # 5. Aggregate the supersampled grid back to original DEM dimensions to calculate "1"s ratio per pixel
    # This step reshapes the 2D matrix to calculate the average block by block
    reshaped = fine_mask.reshape(dem_height, scale, dem_width, scale)
    # Calculate the ratio of sub-pixels covered by the river channel within each macro-pixel (0.0 ~ 1.0)
    coverage = reshaped.mean(axis=(1, 3))
    
    # 6. Apply the desired "in-between" threshold
    # If coverage is greater than or equal to the threshold, classify as river (1), otherwise land (0)
    final_mask = (coverage >= coverage_threshold).astype('uint8')
    
    """
    TODO: we are using polygon not polyline
    # 7. Forcefully burn in the skeleton line to ensure river continuity (double insurance)
    # If a high threshold causes localized disconnected channels, this line forces single-pixel skeleton burn-in
    skeleton_geoms = gdf.geometry.boundary if gdf.geometry.iloc[0].geom_type in ['Polygon', 'MultiPolygon'] else gdf.geometry
    shapes_skeleton = ((geom, 1) for geom in skeleton_geoms)
    mask_skeleton = features.rasterize(
        shapes=shapes_skeleton,
        out_shape=(dem_height, dem_width),
        transform=dem_transform,
        fill=0,
        all_touched=False, 
        dtype='uint8'
    )
    final_mask = np.where(mask_skeleton == 1, 1, final_mask)"""

    # 8. Save the result
    dem_meta.update({
        'driver': 'GTiff',
        'dtype': 'uint8',
        'count': 1,
        'nodata': 0
    })

    with rasterio.open(output_tif, 'w', **dem_meta) as dst:
        dst.write(final_mask, 1)
        
    print(f"--- Hybrid Rasterization Completed ---")
    print(f"Current area threshold: {coverage_threshold * 100}%")

# --- Function Call ---
# Passing 0.3 means: as long as the river covers more than 30% of a pixel's area, it is marked as river.
# You can freely tweak this value (e.g., 0.25, 0.4) until you are satisfied with the river width.
# river_mask_hybrid("./data/test_mask_1.shp", "model_data/2d/dem.tif", "./mask_data/river_mask_05.tif", coverage_threshold=0.5)
river_mask_hybrid("test_mask_1.shp", "dem.tif", "river_mask_05.tif", coverage_threshold=0.5)

--- Hybrid Rasterization Completed ---
Current area threshold: 50.0%
